In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q openai tqdm

In [3]:
!find /content/drive/MyDrive -iname "anime_complete.csv"
!find /content/drive/MyDrive -iname "anime_complete_encoded.csv"

/content/drive/MyDrive/anime_complete.csv
/content/drive/MyDrive/yuran_files/datasets_needed/anime_complete.csv


In [8]:
# ============================================================
# FINAL: Controlled Semantic Tag Extraction for Anime Synopsis
# Uses OpenAI Responses API
# Outputs:
#   1) synopsis_semantic_tags_raw.csv
#   2) synopsis_semantic_tags.npy
# ============================================================

# ----------------------------
# 0. Install / imports
# ----------------------------

import os
import re
import json
import time
import ast
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from openai import OpenAI

# ----------------------------
# 1. Paths
# ----------------------------
folder_path = '/content/drive/MyDrive'
anime_csv_path = f'{folder_path}/anime_complete.csv'
eng_path = f'{folder_path}/engineered data'

Path(eng_path).mkdir(parents=True, exist_ok=True)

raw_save_csv = f'{eng_path}/synopsis_semantic_tags_raw.csv'
final_npy_path = f'{eng_path}/synopsis_semantic_tags.npy'
final_feature_csv = f'{eng_path}/synopsis_semantic_tags_features.csv'

# ----------------------------
# 2. Load data
# ----------------------------
df = pd.read_csv(anime_csv_path)

# Adjust these if your column names differ
ID_COL = "MAL_ID"
SYNOPSIS_COL = "synopsis"

if ID_COL not in df.columns:
    raise ValueError(f"Missing required column: {ID_COL}")
if SYNOPSIS_COL not in df.columns:
    raise ValueError(f"Missing required column: {SYNOPSIS_COL}")

df = df[[ID_COL, SYNOPSIS_COL]].copy()
df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce")
df = df.dropna(subset=[ID_COL]).copy()
df[ID_COL] = df[ID_COL].astype(int)
df[SYNOPSIS_COL] = df[SYNOPSIS_COL].fillna("").astype(str)

print("Rows to process:", len(df))

# ----------------------------
# 3. OpenAI client
# ----------------------------
# In Colab, set your key first, e.g.
# os.environ["OPENAI_API_KEY"] = "sk-..."
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment.")

# ----------------------------
# 4. Controlled taxonomy
#    IMPORTANT:
#    - Genre is NOT extracted by AI because you already have a Genres column.
#    - AI only extracts complementary semantic attributes.
# ----------------------------
TONE_CHOICES = [
    "lighthearted", "dark", "emotional", "comedic", "serious",
    "melancholic", "inspirational", "tense", "wholesome", "chaotic", "unknown"
]

SETTING_CHOICES = [
    "school", "fantasy_world", "modern_city", "historical", "sci_fi_future",
    "supernatural_world", "workplace", "war_zone", "rural_town",
    "virtual_game_world", "post_apocalyptic", "sports_environment",
    "space", "unknown"
]

STORY_TYPE_CHOICES = [
    "character_driven", "plot_driven", "episodic", "romance_driven",
    "action_driven", "mystery_driven", "coming_of_age", "survival",
    "competition", "ensemble_cast", "unknown"
]

PACING_CHOICES = ["slow", "medium", "fast"]

CORE_MOTIF_CHOICES = [
    "friendship", "revenge", "family", "identity", "survival", "sacrifice",
    "ambition", "redemption", "love", "loss", "betrayal", "justice",
    "destiny", "self_discovery", "loyalty", "power", "war", "memory",
    "loneliness", "hope"
]

taxonomy_text = f"""
You must classify each anime synopsis using ONLY the following controlled vocabularies.

tone: {TONE_CHOICES}
setting: {SETTING_CHOICES}
story_type: {STORY_TYPE_CHOICES}
pacing: {PACING_CHOICES}
core_motifs: choose up to 3 from {CORE_MOTIF_CHOICES}

Also output:
- emotional_intensity: integer from 1 to 5
- complexity: integer from 1 to 5

Rules:
- Do NOT invent new labels.
- Do NOT output genre.
- If unclear, choose "unknown" where available.
- Return valid JSON only.
"""

SYSTEM_PROMPT = f"""
You are helping build features for an anime recommendation system.

Your task is to extract structured semantic tags from anime synopses using a CONTROLLED taxonomy.

{taxonomy_text}

Return JSON in exactly this schema:
{{
  "tone": "one label from tone choices",
  "setting": "one label from setting choices",
  "story_type": "one label from story_type choices",
  "pacing": "one label from pacing choices",
  "core_motifs": ["up to 3 labels from core_motif choices"],
  "emotional_intensity": 1,
  "complexity": 1
}}
"""

# ----------------------------
# 5. JSON parsing helpers
# ----------------------------
def extract_json_object(text: str):
    """
    Tries to extract the first JSON object from model output.
    """
    text = text.strip()

    # direct parse
    try:
        return json.loads(text)
    except Exception:
        pass

    # try fenced code block
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        candidate = match.group(0)
        try:
            return json.loads(candidate)
        except Exception:
            pass

    raise ValueError(f"Could not parse JSON from response: {text[:500]}")


def coerce_tag_output(obj):
    """
    Force output into controlled schema.
    """
    tone = obj.get("tone", "unknown")
    setting = obj.get("setting", "unknown")
    story_type = obj.get("story_type", "unknown")
    pacing = obj.get("pacing", "medium")
    core_motifs = obj.get("core_motifs", [])
    emotional_intensity = obj.get("emotional_intensity", 3)
    complexity = obj.get("complexity", 3)

    if tone not in TONE_CHOICES:
        tone = "unknown"
    if setting not in SETTING_CHOICES:
        setting = "unknown"
    if story_type not in STORY_TYPE_CHOICES:
        story_type = "unknown"
    if pacing not in PACING_CHOICES:
        pacing = "medium"

    if not isinstance(core_motifs, list):
        core_motifs = []

    clean_motifs = []
    for m in core_motifs:
        if m in CORE_MOTIF_CHOICES and m not in clean_motifs:
            clean_motifs.append(m)
    core_motifs = clean_motifs[:3]

    try:
        emotional_intensity = int(emotional_intensity)
    except Exception:
        emotional_intensity = 3
    try:
        complexity = int(complexity)
    except Exception:
        complexity = 3

    emotional_intensity = max(1, min(5, emotional_intensity))
    complexity = max(1, min(5, complexity))

    return {
        "tone": tone,
        "setting": setting,
        "story_type": story_type,
        "pacing": pacing,
        "core_motifs": core_motifs,
        "emotional_intensity": emotional_intensity,
        "complexity": complexity,
    }

# ----------------------------
# 6. API call helper
# ----------------------------
# Official OpenAI Python SDK now uses OpenAI() and client.responses.create(...).  [oai_citation:1‡OpenAI Developers](https://developers.openai.com/api/reference/python/?utm_source=chatgpt.com)
MODEL_NAME = "gpt-5.4"

def extract_semantic_tags_from_synopsis(synopsis: str, max_retries: int = 3):
    synopsis = str(synopsis).strip()

    if synopsis == "":
        return {
            "tone": "unknown",
            "setting": "unknown",
            "story_type": "unknown",
            "pacing": "medium",
            "core_motifs": [],
            "emotional_intensity": 3,
            "complexity": 3,
        }

    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=MODEL_NAME,
                input=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"Anime synopsis:\n{synopsis}"}
                ],
            )

            text = response.output_text
            obj = extract_json_object(text)
            return coerce_tag_output(obj)

        except Exception as e:
            if attempt == max_retries - 1:
                return {
                    "tone": "unknown",
                    "setting": "unknown",
                    "story_type": "unknown",
                    "pacing": "medium",
                    "core_motifs": [],
                    "emotional_intensity": 3,
                    "complexity": 3,
                    "error_msg": str(e),
                }
            time.sleep(1.5 * (attempt + 1))

# ----------------------------
# 7. Resume support
# ----------------------------
processed_ids = set()
existing_rows = []

if os.path.exists(raw_save_csv):
    existing_df = pd.read_csv(raw_save_csv)
    if ID_COL in existing_df.columns:
        processed_ids = set(existing_df[ID_COL].dropna().astype(int).tolist())
        existing_rows = existing_df.to_dict(orient="records")
        print(f"Resuming from existing file. Already processed: {len(processed_ids)}")

todo_df = df[~df[ID_COL].isin(processed_ids)].copy()
print("Remaining to process:", len(todo_df))

# ----------------------------
# 8. Batch process and save incrementally
# ----------------------------
results = existing_rows.copy()

SAVE_EVERY = 50   # write partial results every N rows
SLEEP_BETWEEN_CALLS = 0.15

for idx, row in enumerate(tqdm(todo_df.itertuples(index=False), total=len(todo_df))):
    mal_id = int(getattr(row, ID_COL))
    synopsis = getattr(row, SYNOPSIS_COL)

    tags = extract_semantic_tags_from_synopsis(synopsis)

    record = {
        ID_COL: mal_id,
        **tags
    }
    results.append(record)

    if (idx + 1) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(raw_save_csv, index=False)

    time.sleep(SLEEP_BETWEEN_CALLS)

# final save
df_tags = pd.DataFrame(results)
df_tags.to_csv(raw_save_csv, index=False)

print("Saved raw tags to:", raw_save_csv)
print(df_tags.head())

# ----------------------------
# 9. Convert tags to numeric feature matrix
# ----------------------------
df_tags = pd.read_csv(raw_save_csv)

def safe_parse_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        val = ast.literal_eval(x)
        return val if isinstance(val, list) else []
    except Exception:
        return []

df_tags["core_motifs"] = df_tags["core_motifs"].apply(safe_parse_list)

# Ensure same row order as anime_complete.csv
df_merge = df[[ID_COL]].merge(df_tags, on=ID_COL, how="left")

# Fill missing
for col in ["tone", "setting", "story_type", "pacing"]:
    df_merge[col] = df_merge[col].fillna("unknown" if col != "pacing" else "medium")

for col in ["emotional_intensity", "complexity"]:
    df_merge[col] = pd.to_numeric(df_merge[col], errors="coerce").fillna(3).astype(np.float32)

df_merge["core_motifs"] = df_merge["core_motifs"].apply(lambda x: x if isinstance(x, list) else [])

# Multi-hot for motifs
motif_mlb = MultiLabelBinarizer(classes=CORE_MOTIF_CHOICES)
motif_matrix = motif_mlb.fit_transform(df_merge["core_motifs"]).astype(np.float32)

# One-hot for categorical semantic labels
tone_ohe = pd.get_dummies(df_merge["tone"], prefix="tone", dtype=np.float32)
setting_ohe = pd.get_dummies(df_merge["setting"], prefix="setting", dtype=np.float32)
story_ohe = pd.get_dummies(df_merge["story_type"], prefix="story", dtype=np.float32)
pacing_ohe = pd.get_dummies(df_merge["pacing"], prefix="pacing", dtype=np.float32)

# Reindex to ensure fixed column set
tone_cols = [f"tone_{x}" for x in TONE_CHOICES]
setting_cols = [f"setting_{x}" for x in SETTING_CHOICES]
story_cols = [f"story_{x}" for x in STORY_TYPE_CHOICES]
pacing_cols = [f"pacing_{x}" for x in PACING_CHOICES]

tone_ohe = tone_ohe.reindex(columns=tone_cols, fill_value=0)
setting_ohe = setting_ohe.reindex(columns=setting_cols, fill_value=0)
story_ohe = story_ohe.reindex(columns=story_cols, fill_value=0)
pacing_ohe = pacing_ohe.reindex(columns=pacing_cols, fill_value=0)

numeric_part = df_merge[["emotional_intensity", "complexity"]].values.astype(np.float32)

semantic_feature_matrix = np.concatenate([
    motif_matrix,
    tone_ohe.values.astype(np.float32),
    setting_ohe.values.astype(np.float32),
    story_ohe.values.astype(np.float32),
    pacing_ohe.values.astype(np.float32),
    numeric_part
], axis=1).astype(np.float32)

# save aligned features
np.save(final_npy_path, semantic_feature_matrix)

# save aligned feature table too
feature_df = pd.DataFrame(semantic_feature_matrix)
feature_df.insert(0, ID_COL, df_merge[ID_COL].values)
feature_df.to_csv(final_feature_csv, index=False)

print("Saved aligned semantic feature matrix to:", final_npy_path)
print("Saved aligned semantic feature table to:", final_feature_csv)
print("semantic_feature_matrix shape:", semantic_feature_matrix.shape)

# ----------------------------
# 10. Quick sanity check
# ----------------------------
print("\nSanity check:")
print("Input anime rows:", len(df))
print("Aligned tag rows:", len(df_merge))
print("Matrix rows:", semantic_feature_matrix.shape[0])
print("Matrix dims:", semantic_feature_matrix.shape[1])
print(df_merge[[ID_COL, "tone", "setting", "story_type", "pacing", "core_motifs", "emotional_intensity", "complexity"]].head(10))

Rows to process: 17562
Remaining to process: 17562


100%|██████████| 17562/17562 [9:52:30<00:00,  2.02s/it]


Saved raw tags to: /content/drive/MyDrive/engineered data/synopsis_semantic_tags_raw.csv
   MAL_ID          tone           setting        story_type  pacing  \
0       1  lighthearted             space     ensemble_cast  medium   
1       5         tense             space    mystery_driven    fast   
2       6       serious  post_apocalyptic  character_driven    fast   
3       7       serious       modern_city    mystery_driven  medium   
4       8       serious     fantasy_world     action_driven    fast   

                       core_motifs  emotional_intensity  complexity  
0   [friendship, memory, identity]                    3           3  
1  [survival, justice, friendship]                    4           4  
2  [redemption, justice, identity]                    4           4  
3       [identity, justice, power]                    3           3  
4   [sacrifice, justice, ambition]                    4           2  
Saved aligned semantic feature matrix to: /content/drive/MyDrive

To use the OpenAI API, you'll need an API key. If you don't already have one, create one on the [OpenAI platform](https://platform.openai.com/account/api-keys).

In Colab, add the key to the secrets manager under the "🔑" icon in the left panel. Give it the name `OPENAI_API_KEY`.

Then, run the following cell to retrieve it:

In [6]:
from google.colab import userdata

# Set the environment variable OPENAI_API_KEY
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("OPENAI_API_KEY has been set from Colab secrets.")

OPENAI_API_KEY has been set from Colab secrets.


In [9]:
from google.colab import files

# Define the paths to the generated files
eng_path = '/content/drive/MyDrive/engineered data'
raw_save_csv = f'{eng_path}/synopsis_semantic_tags_raw.csv'
final_npy_path = f'{eng_path}/synopsis_semantic_tags.npy'
final_feature_csv = f'{eng_path}/synopsis_semantic_tags_features.csv'

# Download the raw CSV file
print(f'Downloading {raw_save_csv.split("/")[-1]}...')
files.download(raw_save_csv)

# Download the NPY file
print(f'Downloading {final_npy_path.split("/")[-1]}...')
files.download(final_npy_path)

# Download the feature CSV file
print(f'Downloading {final_feature_csv.split("/")[-1]}...')
files.download(final_feature_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>